# 340 · Resonator spectroscopy in parallel — solution

**Level:** advanced · **Topics:** Gen2, SHFQA, spectroscopy, readout, sweeps, calibration · **Type:** fix the bugs

Solution of [exercise.ipynb](exercise.ipynb).

In [ ]:
from laboneq.simple import *

## Device setup

One SHFQA+ with four readout channels. Two are used, each with two qubits.

📖 [Device setup](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/00_device_setup/concepts/index.html)

In [ ]:
qubits = ["q0", "q1", "q2", "q3"]

setup = DeviceSetup("exercise_setup")
setup.add_dataserver(host="localhost", port="8004")
setup.add_instruments(SHFQA(uid="shfqa", address="DEV12001", device_options="SHFQA4"))
setup.add_connections(
    "shfqa",
    create_connection(to_signal="q0/measure", ports="QACHANNELS/0/OUTPUT"),
    create_connection(to_signal="q0/acquire", ports="QACHANNELS/0/INPUT"),
    create_connection(to_signal="q1/measure", ports="QACHANNELS/0/OUTPUT"),
    create_connection(to_signal="q1/acquire", ports="QACHANNELS/0/INPUT"),
    create_connection(to_signal="q2/measure", ports="QACHANNELS/1/OUTPUT"),
    create_connection(to_signal="q2/acquire", ports="QACHANNELS/1/INPUT"),
    create_connection(to_signal="q3/measure", ports="QACHANNELS/1/OUTPUT"),
    create_connection(to_signal="q3/acquire", ports="QACHANNELS/1/INPUT"),
)

## Session

📖 [Session](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/01_session/concepts/index.html)

In [ ]:
session = Session(setup)
session.connect(do_emulation=True)

## The spectroscopy of the user

`spectroscopy(qubits)` builds one experiment that sweeps the readout frequency of all the given qubits at the same time.
Every qubit has its own sweep, 20 MHz wide. The oscillators are set in the calibration of the experiment.

The function itself is fine, there is nothing to change in it.

📖 [Frequency sweep examples](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/03_sections_pulses/tutorials/03_frequency_sweep_examples.html) ·
[Experiment calibration](https://docs.zhinst.com/labone_q_user_manual/core/functionality_and_concepts/05_experiment/concepts/00_experiment_calibration.html) ·
[Resonator spectroscopy](https://docs.zhinst.com/labone_q_user_manual/applications_library/how-to-guides/sources/01_superconducting_qubits/01_workflows/01_resonator_spectroscopy.html)

In [ ]:
LO_FREQUENCY = 7.0e9
SWEEP_START = {"q0": -150e6, "q1": 60e6, "q2": -90e6, "q3": 120e6}

readout_pulse = pulse_library.const(length=2e-6, amplitude=0.1)


def spectroscopy(qubits):
    signals = []
    sweeps = []
    for qubit in qubits:
        signals.append(f"{qubit}_measure")
        signals.append(f"{qubit}_acquire")
        start = SWEEP_START[qubit]
        sweeps.append(
            LinearSweepParameter(uid=f"frequency_{qubit}", start=start, stop=start + 20e6, count=21)
        )

    @dsl.experiment(signals=signals)
    def experiment():
        with dsl.acquire_loop_rt(count=16, acquisition_type=AcquisitionType.SPECTROSCOPY):
            with dsl.sweep(parameter=sweeps):
                for qubit in qubits:
                    with dsl.section(name=f"measure_{qubit}"):
                        dsl.measure(
                            measure_signal=f"{qubit}_measure",
                            acquire_signal=f"{qubit}_acquire",
                            measure_pulse=readout_pulse,
                            integration_length=2e-6,
                            handle=f"{qubit}/result",
                            reset_delay=1e-6,
                        )

        experiment_calibration = dsl.experiment_calibration()
        for qubit, sweep in zip(qubits, sweeps):
            oscillator = Oscillator(frequency=sweep, modulation_type=ModulationType.HARDWARE)
            experiment_calibration[f"{qubit}_measure"] = SignalCalibration(
                oscillator=oscillator,
                local_oscillator=Oscillator(frequency=LO_FREQUENCY),
                range=-10,
            )
            experiment_calibration[f"{qubit}_acquire"] = SignalCalibration(
                oscillator=oscillator,
                local_oscillator=Oscillator(frequency=LO_FREQUENCY),
                range=0,
            )

    exp = experiment()
    for qubit in qubits:
        exp.map_signal(f"{qubit}_measure", f"{qubit}/measure")
        exp.map_signal(f"{qubit}_acquire", f"{qubit}/acquire")
    return exp

## Run it

In spectroscopy mode a readout channel has **one** oscillator, so it can sweep one resonator at a time.
Two qubits on different channels can run together, two qubits on the same channel cannot.

The qubits are split in two groups, each with one qubit per channel: two experiments instead of four.

In [ ]:
data = {}

groups = [["q0", "q2"], ["q1", "q3"]]
for group in groups:
    compiled = session.compile(spectroscopy(group))
    results = session.run(compiled)
    for qubit in group:
        data[qubit] = results[f"{qubit}/result"].data

In [ ]:
# check
for qubit in qubits:
    print(qubit, data[qubit].shape)

**Expected result:**

```
q0 (21,)
q1 (21,)
q2 (21,)
q3 (21,)
```

## Answers

1. `No free hardware oscillators available for signal 'q1_measure'. Missing LRT option?` The frequency sweep in spectroscopy mode uses the hardware oscillator of the readout channel, and a channel has one.
2. One per readout channel: 4 on an SHFQA4, 2 with the wiring of this exercise.
3. Not for this experiment. LRT gives more oscillators per channel (3 on an SHFQA, 6 on an SHFQC), and the oscillator error disappears. But in spectroscopy mode a channel still plays a single envelope, and LabOne Q 26.10 stops with `Only one envelope waveform per physical channel is possible in spectroscopy mode`.
4. No. Two sweeps in series on the same channel fail with the same oscillator error: the oscillators are assigned for the whole experiment, not per sweep.

**Takeaway:** in spectroscopy mode, parallel means one resonator per readout channel. Group the qubits by channel and run one experiment per group.